# Loading Data

In [2]:
import pandas as pd
import numpy as np

In [3]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample_submit = pd.read_csv("sample_submission.csv")

# Add Feature

In [4]:
# adding feature
def add_features(df):
    df = df.copy()
    df['screen_minus_sleep'] = df['daily_screen_time_hours'] - df['sleep_hours']
    df['social_ratio'] = df['social_media_hours'] / (df['daily_screen_time_hours'] + 1e-3)
    df['gaming_ratio'] = df['gaming_hours'] / (df['daily_screen_time_hours'] + 1e-3)
    df['weekend_vs_daily'] = df['weekend_screen_time'] / (df['daily_screen_time_hours'] + 1e-3)
    df['opens_per_notification'] = df['app_opens_per_day'] / (df['notifications_per_day'] + 1)
    df['leisure_hours'] = df['social_media_hours'].fillna(0) + df['gaming_hours'].fillna(0)
    df['n_missing'] = df.isna().sum(axis=1)
    df["social_media_hours_exp"] = df['social_media_hours']**0.3
    df["gaming_hours_exp"] = df['gaming_hours']**0.2
    df["work_study_hours_exp_015"] = df['work_study_hours']**0.15
    return df

In [5]:
# work study hours
train = add_features(train)
test = add_features(test)

# Modeling with LightGBM

In [6]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

In [9]:
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, Pool

In [10]:
# 2. Identify your string columns
string_cols = ["gender", "stress_level","academic_work_impact"]

# 3. Convert string columns to pandas 'category' type
for col in string_cols:
    train[col] = train[col].astype("category")
    test[col] = test[col].astype("category")

In [11]:
# 4. Separate features and target
X = train.drop(columns=["id","addicted_label"])
y = train["addicted_label"]
X_test = test.drop(columns=["id"])

In [12]:
SEED = 42
N_FOLDS = 5
TARGET = 'addicted_label'

In [13]:
# Stratified KFold cross validation
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(X, y))

def run_cv(fit_predict_fn, name):
    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))
    scores = []
    for fold, (tr_idx, va_idx) in enumerate(FOLDS):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]
        va_pred, te_pred = fit_predict_fn(X_tr, y_tr, X_va, y_va)
        oof[va_idx] = va_pred
        test_pred += te_pred / N_FOLDS
        auc = roc_auc_score(y_va, va_pred)
        scores.append(auc)
        print(f'  fold {fold}: AUC = {auc:.5f}')
    total = roc_auc_score(y, oof)
    print(f'{name} | OOF AUC = {total:.5f} (mean {np.mean(scores):.5f} ± {np.std(scores):.5f})')
    return oof, test_pred, total

# Modeling

## Light GBM

In [14]:
# function lgb

def fit_lgb(X_tr, y_tr, X_va, y_va):
    model = lgb.LGBMClassifier(
        objective='binary',
        n_estimators=5000,
        learning_rate=0.03,
        num_leaves=63,
        min_child_samples=60,
        colsample_bytree=0.8,
        subsample=0.8,
        subsample_freq=1,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=SEED,
        n_jobs=-1,
        verbose=-1,
    )
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        eval_metric='auc',
        callbacks=[lgb.early_stopping(200, verbose=False)],
        categorical_feature=string_cols,
    )
    return (model.predict_proba(X_va)[:, 1],
            model.predict_proba(X_test)[:, 1])

In [15]:
oof_lgb, test_lgb, auc_lgb = run_cv(fit_lgb, 'LightGBM')

C:\Users\User\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  fold 0: AUC = 0.96289


C:\Users\User\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  fold 1: AUC = 0.96363


C:\Users\User\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  fold 2: AUC = 0.96409


C:\Users\User\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  fold 3: AUC = 0.96456


C:\Users\User\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  fold 4: AUC = 0.96352
LightGBM | OOF AUC = 0.96374 (mean 0.96374 ± 0.00056)


## XGBoost

In [16]:
def fit_xgb(X_tr, y_tr, X_va, y_va):
    model = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='auc',
        n_estimators=5000,
        learning_rate=0.03,
        max_depth=7,
        min_child_weight=20,
        colsample_bytree=0.8,
        subsample=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        tree_method='hist',
        enable_categorical=True,
        early_stopping_rounds=200,
        random_state=SEED,
        n_jobs=-1,
    )
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    return (model.predict_proba(X_va)[:, 1],
            model.predict_proba(X_test)[:, 1])

In [17]:
oof_xgb, test_xgb, auc_xgb = run_cv(fit_xgb, 'XGBoost')

  fold 0: AUC = 0.96367
  fold 1: AUC = 0.96432
  fold 2: AUC = 0.96448
  fold 3: AUC = 0.96517
  fold 4: AUC = 0.96412
XGBoost | OOF AUC = 0.96435 (mean 0.96435 ± 0.00049)


## Cat Boost

In [ ]:
'''
def cat_prepare(df):
    df = df.copy()
    for c in string_cols:
        df[c] = df[c].cat.add_categories('missing').fillna('missing').astype(str)
    return df

X_cb, X_test_cb = cat_prepare(X), cat_prepare(X_test)

def fit_cat(X_tr, y_tr, X_va, y_va):
    tr_idx, va_idx = X_tr.index, X_va.index
    X_tr, X_va = X_cb.loc[tr_idx], X_cb.loc[va_idx]
    model = CatBoostClassifier(
        iterations=5000,
        learning_rate=0.03,
        depth=7,
        l2_leaf_reg=3.0,
        eval_metric='AUC',
        cat_features=string_cols,
        random_seed=SEED,
        early_stopping_rounds=200,
        verbose=False,
    )
    model.fit(X_tr, y_tr, eval_set=(X_va, y_va))
    return (model.predict_proba(X_va)[:, 1],
            model.predict_proba(X_test_cb)[:, 1])
'''

In [ ]:
# oof_cat, test_cat, auc_cat = run_cv(fit_cat, 'CatBoost')

## Ensemble

In [ ]:
'''
from scipy.stats import rankdata

# rank data used to rank the data at hand. Example, I have data of [1,4,3,5,5], the output would be [1,3,2,4,4]
# as 5 is at rank 4

results = pd.DataFrame({
    'model': ['LightGBM', 'XGBoost', 'CatBoost'],
    'oof_auc': [auc_lgb, auc_xgb, auc_cat],
}).sort_values('oof_auc', ascending=False)
display(results)

oof_mean = (oof_lgb + oof_xgb + oof_cat) / 3
oof_rank = (rankdata(oof_lgb) + rankdata(oof_xgb) + rankdata(oof_cat)) / (3 * len(oof_lgb))

print('Ensamble (media)  OOF AUC:', roc_auc_score(y, oof_mean))
print('Ensamble (rangos) OOF AUC:', roc_auc_score(y, oof_rank))
'''

In [19]:
# test_mean = (test_lgb + test_xgb + test_cat) / 3
test_mean = (test_lgb + test_xgb ) / 2
# test_rank = (rankdata(test_lgb) + rankdata(test_xgb) + rankdata(test_cat)) / (3 * len(test_lgb))

def save_sub(pred, name):
    sub = sample_submit.copy()
    sub[TARGET] = pred
    sub.to_csv(name, index=False)
    print('guardado:', name)

save_sub(test_lgb,  'sub_lgb.csv')
save_sub(test_xgb,  'sub_xgb.csv')
#save_sub(test_cat,  'sub_cat.csv')
save_sub(test_mean, 'sub_ensemble_mean.csv')
# save_sub(test_rank, 'sub_ensemble_rank.csv')

guardado: sub_lgb.csv
guardado: sub_xgb.csv
guardado: sub_ensemble_mean.csv


In [20]:
oof_lgb

array([0.85446908, 0.02381159, 0.46961581, ..., 0.67768598, 0.97997828,
       0.22205707], shape=(691369,))

In [21]:
train.head()

,id,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,...,screen_minus_sleep,social_ratio,gaming_ratio,weekend_vs_daily,opens_per_notification,leisure_hours,n_missing,social_media_hours_exp,gaming_hours_exp,work_study_hours_exp_015
0,0,24.0,NaN,1.83,1.59,2.11,7.46,122.0,38.0,8.63,...,NaN,NaN,NaN,NaN,0.308943,3.42,5,1.198769,1.097184,1.118516
1,1,19.0,5.97,1.08,NaN,3.03,8.22,76.0,19.0,NaN,...,-2.25,0.180874,NaN,NaN,0.246753,1.08,4,1.023357,NaN,1.180909
2,2,18.0,5.09,NaN,NaN,NaN,6.25,134.0,60.0,7.47,...,-1.16,NaN,NaN,1.467295,0.444444,0.00,5,NaN,NaN,NaN
3,3,21.0,6.42,1.26,1.42,3.36,8.85,112.0,94.0,8.66,...,-2.43,0.196231,0.221149,1.348700,0.831858,2.68,1,1.071794,1.072649,1.199364
4,4,26.0,11.20,1.87,2.81,1.95,5.25,NaN,NaN,13.39,...,5.95,0.166949,0.250870,1.195429,NaN,4.68,3,1.206570,1.229536,1.105364


In [29]:
def save_oof(pred, name):
    oof = train.copy()
    oof = oof[["id",TARGET]]
    oof[TARGET] = pred
    oof.to_csv(name, index=False)
    print('saving:', name)

In [30]:
save_oof(oof_lgb,'oof_lgb.csv')
save_oof(oof_xgb,'oof_xgb.csv')

saving: oof_lgb.csv
saving: oof_xgb.csv
